# CyberGuardian RL Training Notebook
Open this directly from GitHub, and run all to train your deep RL agents natively on Colab.

In [ ]:
!git clone https://github.com/GiGiKoneti/CyberGuardian-AI.git
import os
os.chdir('CyberGuardian-AI/backend')
!pip install -r requirements.txt --quiet

In [ ]:
import gymnasium as gym
import numpy as np
import torch as th
from src.training.teacher_ppo import TeacherGuidedPPO
from src.environment.cyber_env import CyberSecurityEnv
from src.agents.llm_blue_agent import LLMBlueAgent

class ManualFlattenAction(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.action_space = gym.spaces.MultiDiscrete([20, 6, 20, 6])

    def step(self, action):
        dict_action = {'blue_action': action[0:2], 'red_action': action[2:4]}
        obs, reward, terminated, truncated, info = self.env.step(dict_action)
        if isinstance(reward, dict): reward = float(sum(reward.values()))
        else: reward = float(reward)
        return obs, reward, terminated, truncated, info

class TeacherWrapper:
    def __init__(self, teacher_agent):
        self.teacher_agent = teacher_agent
    
    def predict(self, obs, state=None, episode_start=None, deterministic=False):
        # LLMBlueAgent has a predict method as shown in diagnostic
        blue_action, _ = self.teacher_agent.predict(obs)
        # Pad with dummy red actions
        return np.array([blue_action[0], blue_action[1], 0, 0], dtype=np.int64), None

print('Initializing Environment with 20 nodes...')
env = ManualFlattenAction(CyberSecurityEnv(num_hosts=20))
teacher = TeacherWrapper(LLMBlueAgent())

# Re-enabling verbose output and tensorboard logging for the full run
model = TeacherGuidedPPO('MultiInputPolicy', env, teacher_agent=teacher, verbose=1, tensorboard_log='./ppo_metrics/')

print('Starting production training run (100k steps)...')
model.learn(total_timesteps=100000)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil

# Save trained model weights
# We exclude 'teacher_agent' because it contains non-serializable objects (like API clients/generators)
model.save('/content/drive/MyDrive/blue_ppo_bot.zip', exclude=['teacher_agent'])

# Archive the training metrics
shutil.make_archive('/content/drive/MyDrive/tensorboard_metrics', 'zip', './ppo_metrics/')

print('Successfully preserved Blue Agent weights (excluding teacher) and Tensorboard configurations to Drive!')